# GGUF Export — GPU + Unsloth

Reloads the fine-tuned adapter from HuggingFace Hub, merges LoRA weights,
and pushes a quantised GGUF directly to a HF repo.

**Requires:** `unsloth-gguf` conda env, NVIDIA GPU, HF write token.

In [1]:
HF_USERNAME     = "CopyleftCultivars"
HF_ADAPTER_REPO = f"{HF_USERNAME}/gemma4-natural-farming-lora"
HF_GGUF_REPO    = f"{HF_USERNAME}/gemma4-natural-farming-gguf"
MAX_SEQ_LENGTH  = 1024
print(f"Adapter : {HF_ADAPTER_REPO}")
print(f"GGUF out: {HF_GGUF_REPO}")

Adapter : CopyleftCultivars/gemma4-natural-farming-lora
GGUF out: CopyleftCultivars/gemma4-natural-farming-gguf


In [2]:
from huggingface_hub import login
import getpass
token = getpass.getpass("HuggingFace write token: ")
login(token=token)
print("Logged in.")

HuggingFace write token:  ········


Logged in.


In [3]:
# Reload base model + LoRA adapter from HF Hub in one shot.
# Unsloth detects adapter_config.json and loads base + LoRA automatically.
# Do NOT use PeftModel.from_pretrained — it double-wraps and breaks GGUF export.
import os, torch
from unsloth import FastModel
from unsloth.chat_templates import get_chat_template

os.environ["PYTORCH_ALLOC_CONF"] = "expandable_segments:True"

model, tokenizer = FastModel.from_pretrained(
    model_name=HF_ADAPTER_REPO,
    max_seq_length=MAX_SEQ_LENGTH,
    load_in_4bit=True,
    dtype=None,
)
tokenizer = get_chat_template(tokenizer, chat_template="gemma-it")
print(f"Loaded: {HF_ADAPTER_REPO}")

ImportError: libnccl.so.2: cannot open shared object file: No such file or directory

In [4]:
# Patch subprocess so the real llama.cpp error prints if conversion fails,
# instead of being swallowed by Unsloth's capture_output=True.
import subprocess
_orig_run = subprocess.run
def _loud_run(args, **kwargs):
    check = kwargs.pop("check", False)
    kwargs.pop("capture_output", None)
    result = _orig_run(args, capture_output=True, **kwargs)
    if result.returncode != 0:
        decode = lambda b: b.decode(errors="replace") if isinstance(b, bytes) else (b or "")
        print("\n=== llama.cpp STDOUT ===")
        print(decode(result.stdout)[-4000:] or "(empty)")
        print("=== llama.cpp STDERR ===")
        print(decode(result.stderr)[-4000:] or "(empty)")
    if check and result.returncode != 0:
        raise subprocess.CalledProcessError(result.returncode, args, result.stdout, result.stderr)
    return result
subprocess.run = _loud_run
print("Subprocess patch active — llama.cpp errors will print here if conversion fails.")

Subprocess patch active — llama.cpp errors will print here if conversion fails.


In [ ]:
# Merge LoRA into base model, convert to GGUF, push to HF Hub.
# temporary_location="/tmp" keeps all intermediate files out of the
# working directory (avoids quota issues on Kaggle/Colab if reused there).
model.push_to_hub_gguf(
    HF_GGUF_REPO,
    tokenizer,
    quantization_method="q4_k_m",
    temporary_location="/tmp",
)
print(f"Done — https://huggingface.co/{HF_GGUF_REPO}")